# Logistic Regression on Phishing Datasets
This notebook applies **Logistic Regression** and **Feature Scaling** to evaluate model performance on phishing datasets for Week 4 of the Capstone.

In [1]:
import pandas as pd
import numpy as np
from pandas.api.types import is_numeric_dtype
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import warnings

warnings.filterwarnings('ignore')

In [2]:
def evaluate_logistic_regression(file_path, target_col, cols_to_drop):
    print(f"\n{'='*60}\nEvaluating Logistic Regression on {file_path}\n{'='*60}")
    
    # 1. Load and Clean Data
    df = pd.read_csv(file_path)
    df.drop(columns=[c for c in cols_to_drop if c in df.columns], inplace=True)
    
    # Handle missing values safely
    for col in df.columns:
        if is_numeric_dtype(df[col]):
            df[col].fillna(df[col].median(), inplace=True)
        else:
            df[col].fillna(df[col].mode()[0], inplace=True)
            
    # Separate features and target
    X = df.drop(columns=[target_col])
    y = df[target_col]
    
    # Encode target variable if it contains text
    if y.dtype == 'object' or y.dtype == 'string':
        encoder = LabelEncoder()
        y = encoder.fit_transform(y)
    
    # Identify Categorical and Continuous Features
    categorical_features = [col for col in X.columns if X[col].dtype == 'object' or X[col].nunique() < 10]
    continuous_features = [col for col in X.columns if col not in categorical_features]
    
    # 2. Preprocessing & Feature Scaling
    # Using StandardScaler to scale continuous features to have mean=0 and variance=1
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), continuous_features),
            ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
        ])
    
    X_processed = preprocessor.fit_transform(X)
    
    # Split the data
    X_train, X_test, y_train, y_test = train_test_split(X_processed, y, test_size=0.2, random_state=42)
    
    # 3. Model Definition and Fitting
    lr = LogisticRegression(max_iter=1000, random_state=42)
    lr.fit(X_train, y_train)
    
    # 4. Evaluation
    y_pred = lr.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f"Accuracy: {acc:.4f}\n")
    print("Classification Report:\n")
    print(classification_report(y_test, y_pred))


## Analysis of `StealthPhisher2025(2).csv`

In [3]:
stealth_file = 'StealthPhisher2025(2).csv'
cols_to_drop_stealth = ['URL', 'Domain', 'TLD', 'HasTitle', 'Title']
evaluate_logistic_regression(stealth_file, 'Label', cols_to_drop_stealth)


Evaluating Logistic Regression on StealthPhisher2025(2).csv
Accuracy: 0.9977

Classification Report:

              precision    recall  f1-score   support

           0       1.00      1.00      1.00     32081
           1       1.00      1.00      1.00     35269

    accuracy                           1.00     67350
   macro avg       1.00      1.00      1.00     67350
weighted avg       1.00      1.00      1.00     67350



## Analysis of `PhiUSIIL_Phishing_URL_Dataset(1).csv`

In [4]:
phiusiil_file = 'PhiUSIIL_Phishing_URL_Dataset(1).csv'
cols_to_drop_phiusiil = ['FILENAME', 'URL', 'Domain', 'TLD', 'Title']
evaluate_logistic_regression(phiusiil_file, 'label', cols_to_drop_phiusiil)


Evaluating Logistic Regression on PhiUSIIL_Phishing_URL_Dataset(1).csv
Accuracy: 0.9999

Classification Report:

              precision    recall  f1-score   support

           0       1.00      1.00      1.00     20124
           1       1.00      1.00      1.00     27035

    accuracy                           1.00     47159
   macro avg       1.00      1.00      1.00     47159
weighted avg       1.00      1.00      1.00     47159

